# Variabilidad del servicio y espera en una cola FCFS
**Cuaderno complementario de la monografía — sin salidas de ejecución guardadas.**

Pregunta: ¿cómo cambia la espera media cuando varía el servicio, manteniendo la tasa de llegada y su media? Esta demostración hipotética compara M/D/1, M/M/1 y M/H₂/1. No representa datos de una organización ni valida un servicio real. No se ha comprobado aquí una ejecución completa en Colab; las tablas y figuras se producirán al ejecutar el código, no son resultados anticipados.

## Uso en Google Colab
1. Abrir https://colab.research.google.com/ y elegir **Archivo → Subir notebook**; seleccionar este `.ipynb`.
2. Usar un entorno Python con CPU y **Entorno de ejecución → Ejecutar todas**. Se requieren NumPy, pandas, Matplotlib, SciPy e IPython (habituales en Colab); no hay instalaciones automáticas. Si falta alguno, detenerse y revisar el entorno.
3. Revisar las pruebas, diagnósticos, tablas y figuras. Guardar una copia con las salidas efectivamente obtenidas; al cambiar parámetros, reiniciar el entorno y ejecutar todo nuevamente.
4. Para entregar al docente, guardar en Drive y usar **Compartir** con permiso de lector según las reglas de la institución. Comprobar el acceso al enlace. Este archivo no publica nada ni crea un enlace automáticamente.

El presupuesto es de **16 800 000 entidades**: puede requerir varios minutos. Se conservan métricas y conteos compactos por réplica, no todas las trayectorias. Autorizar Drive en la sección de almacenamiento antes del cálculo; los productos se guardan en una carpeta exclusiva por ejecución. No se redacta todavía la sección de resultados de la monografía.

## 1. Modelo y fuentes
Una cola abierta, un servidor, capacidad ilimitada y disciplina FCFS (primero en llegar, primero en ser atendido); sin abandonos, prioridades ni interrupciones. Llegadas Poisson, servicios independientes e idénticamente distribuidos e independientes de las llegadas. Se inicia vacío y se sigue cada entidad hasta completar su servicio.

- [Virtamo: cola M/G/1 y Pollaczek–Khinchine](https://ecs.syr.edu/faculty/fawcett/handouts/CSE681summer/Lecture10/queueNotes/E_lect11.pdf): referencia analítica.
- [Banks y colaboradores: análisis de salidas mediante réplicas](https://dmnicol.web.engr.illinois.edu/ece541/slides/output-analysis.pdf): diapositivas del capítulo 11, sin año ni edición verificados.
- Diseño exacto del trabajo: `monografia/secciones/05_metodologia.tex` y sección 5 de `METODOLOGIA_MONOGRAFIA.md`, entregados con el proyecto. Los tamaños, mezcla y tolerancias son decisiones propias, no recomendaciones numéricas atribuidas a esas fuentes.

La media del servicio es $m=1$ minuto y $\lambda=0{,}8$ entidades/minuto; $\rho=\lambda m=0{,}8<1$. Pollaczek–Khinchine predice $w_\star=\lambda E[S^2]/[2(1-\rho)]$. Son referencias **analíticas**, no observaciones: 2, 4 y 8 minutos para los tres escenarios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t
from IPython.display import display

## 2. Configuración y teoría antes de generar datos
Los identificadores fijos son 0: constante, 1: exponencial y 2: H₂. En H₂ se elige una exponencial de **media** $m/2$ con probabilidad 0,8 o de **media** $3m$ con probabilidad 0,2. Por ello $E[S]=m$, $E[S^2]=4m^2$, $\operatorname{Var}(S)=3m^2$ y $C_s^2=3$. Los otros valores de $C_s^2$ son 0 y 1. NumPy recibe medias en `scale`, no tasas.

Cada realización tiene 140 000 entidades. La ventana principal es `W[20000:120000]` y la alternativa `W[40000:140000]`: 100 000 observaciones cada una y 80 000 compartidas. **No se reinicia la cola** al comenzar ninguna ventana.

In [ ]:
from math import isfinite

CONFIG = {'lambda': 0.8, 'm': 1.0, 'replicas': 40, 'entidades': 140000,
          'descarte': 20000, 'alternativo': 40000, 'medicion': 100000,
          'semilla': 2026, 'meta_precision': 0.05, 'tolerancia': 0.02}
ESCENARIOS = {0: 'M/D/1', 1: 'M/M/1', 2: 'M/H₂/1'}

def validar_configuracion(c):
    """Rechazar diseños inválidos antes de generar entradas o archivos."""
    for clave in ('lambda', 'm', 'meta_precision', 'tolerancia'):
        v = c[clave]
        if isinstance(v, bool) or not isinstance(v, (int, float)) or not isfinite(v) or v <= 0:
            raise ValueError(f'{clave}: se requiere un número positivo finito.')
    if c['lambda'] * c['m'] >= 1:
        raise ValueError('Estabilidad: se requiere lambda × m < 1.')
    for clave in ('replicas', 'entidades', 'descarte', 'alternativo', 'medicion', 'semilla'):
        if type(c[clave]) is not int or c[clave] < 0:
            raise ValueError(f'{clave}: se requiere un entero no negativo.')
    if c['replicas'] < 2 or c['medicion'] == 0:
        raise ValueError('Se requieren al menos dos réplicas y medición positiva.')
    if not c['descarte'] < c['alternativo']:
        raise ValueError('El descarte alternativo debe superar al principal.')
    if c['alternativo'] + c['medicion'] > c['entidades']:
        raise ValueError('Las ventanas deben caber en el total de entidades.')

def momentos_teoricos(escenario, c):
    if escenario not in ESCENARIOS:
        raise ValueError('Identificador de escenario desconocido: usar 0, 1 o 2.')
    m = c['m']
    segundo = (1, 2, 4)[escenario] * m**2
    return {'media_servicio_teorica': m, 'segundo_momento': segundo,
            'varianza_servicio_teorica': segundo-m**2,
            'cs2': segundo/m**2-1,
            'espera_teorica': c['lambda']*segundo/(2*(1-c['lambda']*m))}

validar_configuracion(CONFIG)

In [ ]:
teoria = pd.DataFrame([{'escenario_id': e, 'modelo': nombre,
                        **momentos_teoricos(e, CONFIG)} for e, nombre in ESCENARIOS.items()])
print('Referencias poblacionales y analíticas; no son resultados simulados.')
display(teoria)

## 3. Generación reproducible de las entradas
Para cada réplica se deriva `SeedSequence([2026, escenario_id, replica_id]).spawn(3)`: flujos separados para llegadas, componentes y servicios. Se reserva el flujo de componentes incluso cuando no se usa. No hay números aleatorios comunes entre escenarios. Solo las dos ventanas dentro de una misma realización están apareadas. La reproducibilidad depende también de las versiones registradas del software.

In [ ]:
def generar_entradas(escenario, replica, c):
    validar_configuracion(c)
    if escenario not in ESCENARIOS or type(replica) is not int or not 0 <= replica < c['replicas']:
        raise ValueError('Escenario o índice de réplica inválido.')
    semillas = np.random.SeedSequence([c['semilla'], escenario, replica]).spawn(3)
    llegadas_rng, componente_rng, servicio_rng = [np.random.default_rng(s) for s in semillas]
    n, m = c['entidades'], c['m']
    llegadas = np.cumsum(llegadas_rng.exponential(scale=1/c['lambda'], size=n))
    if escenario == 0:
        servicios = np.full(n, m)
    elif escenario == 1:
        servicios = servicio_rng.exponential(scale=m, size=n)
    else:
        medias = np.where(componente_rng.random(n) < 0.8, m/2, 3*m)
        servicios = servicio_rng.exponential(scale=medias)
    return llegadas, servicios

## 4. Núcleo cronológico FCFS
Con $A_i$ llegada y $S_i$ servicio: $B_i=\max(A_i,D_{i-1})$, $D_i=B_i+S_i$, $W_i=B_i-A_i$ y $T_i=D_i-A_i$, con salida previa inicial cero. El núcleo usa solo la biblioteca estándar; puede probarse independientemente de los paquetes científicos. Rechaza entradas no finitas, negativas o desordenadas y no modifica las entradas. Los índices conservan el orden de llegada, incluso con empates.

In [ ]:
from math import isfinite

def simular_fcfs(llegadas, servicios):
    """Devolver listas inicios B, salidas D, esperas W y totales T, en minutos.

    Acepta iterables numéricos de igual longitud, incluso vacíos. El sistema
    comienza vacío; las llegadas deben estar ordenadas y ser no negativas.
    """
    try:
        a = [float(x) for x in llegadas]
        s = [float(x) for x in servicios]
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError('Llegadas y servicios deben ser números finitos.') from error
    if len(a) != len(s):
        raise ValueError('Llegadas y servicios deben tener igual longitud.')
    if any(not isfinite(x) or x < 0 for x in a + s):
        raise ValueError('Llegadas y servicios deben ser finitos y no negativos.')
    if any(a[i] < a[i-1] for i in range(1, len(a))):
        raise ValueError('Las llegadas deben estar ordenadas cronológicamente.')
    resultado = dict(inicios=[], salidas=[], esperas=[], totales=[])
    previa = 0.0
    for llegada, servicio in zip(a, s):
        inicio = max(llegada, previa)
        salida = inicio + servicio
        if not isfinite(salida):
            raise ValueError('Desbordamiento en los tiempos de salida.')
        resultado['inicios'].append(inicio)
        resultado['salidas'].append(salida)
        resultado['esperas'].append(inicio-llegada)
        resultado['totales'].append(salida-llegada)
        previa = salida
    return resultado

## 5. Verificaciones deterministas antes del experimento
Las pruebas siguientes comprueban casos con y sin espera, invariantes y rechazo de datos inválidos. La referencia analítica 2/4/8 se verifica con una configuración de prueba fija; no exige esos mismos valores si se editan los parámetros del experimento. No prueban por sí solas la ejecución científica completa ni la ausencia de sesgo de inicialización.

In [ ]:
r = simular_fcfs([0, 0.5, 2], [1, 1, 1])
assert r == dict(inicios=[0, 1, 2], salidas=[1, 2, 3],
                 esperas=[0, 0.5, 0], totales=[1, 1.5, 1])
r = simular_fcfs([0, 2, 4], [1, 1, 1])
assert r['esperas'] == [0, 0, 0] and r['salidas'] == [1, 3, 5]
a, s = [0, 0, 0.5, 9], [0, 2, 3, 1]
r = simular_fcfs(a, s)
for i in range(len(a)):
    assert r['esperas'][i] >= 0
    assert r['totales'][i] == r['esperas'][i] + s[i]
    assert r['inicios'][i] >= a[i]
    assert i == 0 or r['inicios'][i] >= r['salidas'][i-1]
for a, s in [([1, 0], [1, 1]), ([0], [-1]), ([0], [float('nan')])]:
    try:
        simular_fcfs(a, s)
    except ValueError:
        pass
    else:
        raise AssertionError('Una entrada inválida no fue rechazada.')
config_referencia = {'lambda': 0.8, 'm': 1.0}
for e, esperado in enumerate((2, 4, 8)):
    assert abs(momentos_teoricos(e, config_referencia)['espera_teorica']-esperado) < 1e-12
print('Pruebas deterministas aprobadas; esto no equivale a ejecutar el experimento.')

## Almacenamiento: autorizar antes del experimento
Por defecto se guardan resultados en `MyDrive/Monografia_MyS/resultados/<id UTC + UUID>/`.
Ejecutar la siguiente celda solicita el permiso nativo de Colab: usted decide si autoriza.
No se leen credenciales ni se enumeran otras carpetas. Cada ejecución crea una carpeta nueva,
sin reutilizar resultados previos. Fuera de Colab, Drive produce un error explícito.
Para elegir almacenamiento local, cambiar conscientemente `GUARDAR_EN_DRIVE = False`;
en Colab ese modo es transitorio. `SUBCARPETA` debe ser relativa y no contener `..`.
La configuración y el estado se escriben antes de calcular. Cada réplica conserva métricas
y conteos CDF; una interrupción deja datos **parciales**, nunca un experimento completo.
No reejecutar celdas aisladas: reiniciar y ejecutar en orden para obtener una carpeta nueva.

In [ ]:
from pathlib import Path
import csv
import json
from datetime import datetime, timezone
from uuid import uuid4

def crear_ejecucion(raiz, subcarpeta):
    """Crear una carpeta exclusiva, confinada a la raíz elegida."""
    relativa = Path(subcarpeta)
    if not subcarpeta or relativa.is_absolute() or '..' in relativa.parts or relativa == Path('.'):
        raise ValueError('La subcarpeta debe ser relativa, no vacía y sin ..')
    raiz = Path(raiz).resolve(strict=True)
    base = (raiz/relativa).resolve()
    if not base.is_relative_to(raiz) or base == raiz:
        raise ValueError('El destino debe permanecer bajo la raíz elegida.')
    identificador = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_' + uuid4().hex
    destino = base/identificador
    destino.mkdir(parents=True, exist_ok=False)
    return destino

def ruta_archivo(destino, nombre):
    """Aceptar solo nombres simples y rechazar enlaces que escapen del destino."""
    if not nombre or Path(nombre).name != nombre or nombre in ('.', '..'):
        raise ValueError('Se requiere un nombre de archivo simple.')
    destino = Path(destino).resolve(strict=True)
    ruta = destino/nombre
    if ruta.is_symlink() or ruta.resolve().parent != destino:
        raise ValueError('El archivo debe permanecer dentro de esta ejecución.')
    return ruta

def agregar_csv(destino, nombre, filas):
    """Anexar datos derivados, conservando una única cabecera por archivo."""
    ruta = ruta_archivo(destino, nombre)
    filas = list(filas)
    if not filas:
        return
    campos = list(filas[0])
    if any(list(fila) != campos for fila in filas):
        raise ValueError('Todas las filas deben tener los mismos campos ordenados.')
    existe = ruta.exists()
    if existe:
        with ruta.open(newline='', encoding='utf-8') as archivo:
            if next(csv.reader(archivo), None) != campos:
                raise ValueError('La cabecera existente no coincide.')
    with ruta.open('a' if existe else 'x', newline='', encoding='utf-8') as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=campos)
        if not existe:
            escritor.writeheader()
        escritor.writerows(filas)
        archivo.flush()

def guardar_estado(destino, metadatos, estado, replicas_completadas):
    datos = dict(metadatos, estado=estado, replicas_completadas=replicas_completadas,
                 actualizado_utc=datetime.now(timezone.utc).isoformat())
    ruta_archivo(destino, 'metadata.json').write_text(
        json.dumps(datos, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
import sys
import importlib.util
from importlib.metadata import version

GUARDAR_EN_DRIVE = True
SUBCARPETA = 'Monografia_MyS/resultados'
try:
    es_colab = importlib.util.find_spec('google.colab') is not None
except (ModuleNotFoundError, ValueError):
    es_colab = False
validar_configuracion(CONFIG)
# Validar antes de pedir permiso o crear carpetas.
if not SUBCARPETA or Path(SUBCARPETA).is_absolute() or '..' in Path(SUBCARPETA).parts or Path(SUBCARPETA) == Path('.'):
    raise ValueError('La subcarpeta debe ser relativa, no vacía y sin ..')
if GUARDAR_EN_DRIVE:
    if not es_colab:
        raise RuntimeError('Drive requiere Google Colab. Elegir conscientemente GUARDAR_EN_DRIVE = False para modo local.')
    from google.colab import drive
    drive.mount('/content/drive')
    raiz = Path('/content/drive/MyDrive')
else:
    raiz = Path.cwd()
    print('Modo local elegido: almacenamiento transitorio si se usa Colab; no se guarda en Drive.')
destino = crear_ejecucion(raiz, SUBCARPETA)
metadatos = {'inicio_utc': datetime.now(timezone.utc).isoformat(),
             'entorno': 'Google Colab' if es_colab else 'Python fuera de Colab',
             'almacenamiento': 'Drive' if GUARDAR_EN_DRIVE else 'local',
             'python': sys.version, 'configuracion': dict(CONFIG), 'escenarios': ESCENARIOS,
             'semillas': 'SeedSequence([semilla, escenario_id, replica_id]).spawn(3)',
             'orden_flujos': ['llegadas', 'componente', 'servicio'],
             'versiones': {p: version(p) for p in ('numpy', 'pandas', 'matplotlib', 'scipy', 'ipython')}}
agregar_csv(destino, 'config.csv', [CONFIG])
guardar_estado(destino, metadatos, 'en progreso; datos parciales', 0)
print(f'Destino exclusivo comprobado antes del cálculo: {destino}')

## 6. Experimento de presupuesto fijo
Una sola simulación por réplica alimenta ambas ventanas. Se agregan inmediatamente sus medias y los momentos empíricos del servicio (sobre las 140 000 entradas, varianza muestral con `ddof=1`). Esos momentos son diagnósticos, no se exige igualdad exacta con la población. No se almacenan todas las entidades de todas las réplicas ni se reintenta para conseguir diagnósticos favorables.

In [ ]:
UMBRALES = np.linspace(0, 100*CONFIG['m'], 321)

def ejecutar_experimento(c):
    validar_configuracion(c)
    filas = []
    conteos_totales = {e: np.zeros(len(UMBRALES), dtype=np.int64) for e in ESCENARIOS}
    if ruta_archivo(destino, 'metricas_replicas.csv').exists():
        raise RuntimeError('Esta ejecución ya tiene réplicas; crear una carpeta nueva antes de recalcular.')
    for e, nombre in ESCENARIOS.items():
        for replica in range(c['replicas']):
            a, s = generar_entradas(e, replica, c)
            trayectoria = simular_fcfs(a, s)
            w = np.asarray(trayectoria['esperas'])
            b, bp, n = c['descarte'], c['alternativo'], c['medicion']
            y, yp = float(w[b:b+n].mean()), float(w[bp:bp+n].mean())
            filas.append({'escenario_id': e, 'modelo': nombre, 'replica_id': replica,
                          'Y': y, 'Y_alternativa': yp, 'delta': yp-y,
                          'media_servicio': float(s.mean()),
                          'varianza_servicio': float(s.var(ddof=1)),
                          **c})
            conteos = np.searchsorted(np.sort(w[b:b+n]), UMBRALES, side='right')
            agregar_csv(destino, 'cdf_replicas.csv', [
                {'escenario_id': e, 'replica_id': replica, 'umbral_minutos': float(u),
                 'conteo': int(v), 'denominador': n, 'estado': 'parcial'}
                for u, v in zip(UMBRALES, conteos)])
            agregar_csv(destino, 'metricas_replicas.csv', [dict(filas[-1], estado='parcial')])
            conteos_totales[e] += conteos
            guardar_estado(destino, metadatos, 'en progreso; datos parciales', len(filas))
            del a, s, trayectoria, w
        print(f'{nombre}: {c["replicas"]} réplicas terminadas.')
    return pd.DataFrame(filas), conteos_totales

metricas, conteos_totales = ejecutar_experimento(CONFIG)
print('Métricas agregadas por réplica; primeras filas:')
display(metricas.head())

## 7. Estimaciones e incertidumbre
La unidad estadística es la **media de réplica**, no cada espera correlacionada. Para $K$ medias independientes se usa $\bar Y\pm t_{0.975,K-1}s_Y/\sqrt K$, con varianza muestral (`ddof=1`). El intervalo del 95 % es aproximado; no elimina sesgo común de inicio vacío ni incertidumbre del modelo.

Se informa error relativo **con signo**, $(\bar Y-w_\star)/w_\star$, y la inclusión descriptiva de la referencia en el intervalo. No se exige que toda ejecución contenga la teoría. La meta $h/\bar Y\leq0{,}05$ puede fallar. Para $\Delta=Y'-Y$ se usa otro intervalo entre réplicas; el diagnóstico $|\bar\Delta|+h_\Delta\leq0{,}02w_\star$ puede fallar y **no verifica estacionariedad** aunque sea favorable. Las ventanas comparten 80 000 entidades; no se tratan como independientes.

In [ ]:
def resumir(metricas, c):
    filas = []
    for e, grupo in metricas.groupby('escenario_id', sort=True):
        k = len(grupo)
        if k != c['replicas']:
            raise ValueError('Número inesperado de réplicas en el resumen.')
        critico = float(t.ppf(0.975, k-1))
        media = float(grupo['Y'].mean())
        varianza = float(grupo['Y'].var(ddof=1))
        h = critico * (varianza/k)**0.5
        delta = float(grupo['delta'].mean())
        var_delta = float(grupo['delta'].var(ddof=1))
        hd = critico * (var_delta/k)**0.5
        referencia = momentos_teoricos(e, c)['espera_teorica']
        relativo = h/media if media > 0 else float('inf')
        filas.append({'escenario_id': e, 'modelo': ESCENARIOS[e], 'K': k,
                      'media_espera': media, 'varianza_entre_replicas': varianza,
                      'h_95': h, 'limite_inferior': media-h, 'limite_superior': media+h,
                      'espera_teorica': referencia, 'error_relativo_con_signo': (media-referencia)/referencia,
                      'referencia_en_intervalo': bool(media-h <= referencia <= media+h),
                      'semiamplitud_relativa': relativo,
                      'precision_diagnostico': bool(relativo <= c['meta_precision']),
                      'media_delta': delta, 'varianza_delta': var_delta, 'h_delta_95': hd,
                      'tolerancia_minutos': c['tolerancia']*referencia,
                      'inicializacion_diagnostico': bool(abs(delta)+hd <= c['tolerancia']*referencia)})
    return pd.DataFrame(filas)

resumen = resumir(metricas, CONFIG)
display(resumen)
print('Diagnósticos empíricos del servicio frente a momentos poblacionales:')
display(metricas.groupby('escenario_id')[['media_servicio', 'varianza_servicio']].mean().join(
    teoria.set_index('escenario_id')[['media_servicio_teorica', 'varianza_servicio_teorica']]))
print('Interpretar los indicadores como diagnósticos, no como garantía de ausencia de sesgo.')

## 8. Cuatro figuras y su procedencia
1. **Servicios:** distribuciones teóricas acumuladas (CDF), no muestras simuladas. El servicio constante tiene un átomo de masa 1 en m, no una densidad continua. Misma media no significa misma cola.
2. **Espera media:** medias de réplica e IC Monte Carlo aproximados del 95 %, frente a marcas analíticas; no se recortan los intervalos en cero.
3. **Distribución de esperas:** CDF descriptiva de la ventana principal de todas las réplicas. Grilla fija de 321 umbrales equiespaciados entre 0 y 100m minutos; conteos exactos `searchsorted(side='right')` mientras cada réplica está en memoria. Se exportan conteos por réplica, suma y denominador K×n. La curva entre umbrales es una representación discretizada, no la ECDF completa. Se informa masa por encima del último umbral y no se cierra artificialmente en 1. Las esperas están correlacionadas: no hay IC iid ni certeza de cuantiles.
4. **Inicialización:** diferencia apareada e IC del 95 % frente a ±2 % de la teoría. No demuestra estacionariedad ni ausencia de sesgo compartido.
Las tres últimas figuras proceden de esta ejecución; la primera es analítica. Colores discretos, marcas y estilos consistentes permiten distinguir escenarios también sin color. Media, cola y sensibilidad inicial responden preguntas diferentes. No se interpreta solapamiento de IC como prueba entre modelos.

In [ ]:
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.edgecolor': '0.3', 'axes.linewidth': 0.6, 'font.size': 10})
COLORES = ['#356A96', '#B26449', '#54836B']
MARCAS = ['o', 's', '^']
ESTILOS = ['-', '--', ':']
x = np.arange(len(resumen))
fig_servicios, ax = plt.subplots(figsize=(7, 4), layout='constrained')
m = CONFIG['m']
u = np.linspace(0, 12*m, 481)
ax.step([0, m, 12*m], [0, 1, 1], where='post', color=COLORES[0], label='M/D/1: átomo en m')
ax.plot(m, 0, 'o', color=COLORES[0], markerfacecolor='white')
ax.plot(m, 1, 'o', color=COLORES[0])
ax.plot(u, 1-np.exp(-u/m), color=COLORES[1], linestyle='--', label='M/M/1')
ax.plot(u, 1-0.8*np.exp(-u/(m/2))-0.2*np.exp(-u/(3*m)),
        color=COLORES[2], linestyle=':', label='M/H₂/1')
ax.set(xlabel='Servicio (minutos)', ylabel='Probabilidad acumulada F(s)',
       title='Distribuciones teóricas del servicio', ylim=(-0.03, 1.03))
ax.legend(fontsize=8)
plt.show()

fig_medias, ax = plt.subplots(figsize=(7, 4), layout='constrained')
for i, fila in resumen.reset_index(drop=True).iterrows():
    ax.errorbar(i-0.06, fila['media_espera'], yerr=fila['h_95'], fmt=MARCAS[i],
                color=COLORES[i], capsize=4, linewidth=0.8,
                label='Simulación: IC aproximado 95 %' if i == 0 else None)
    ax.scatter(i+0.06, fila['espera_teorica'], marker='x', color='0.2',
               label='Referencia analítica' if i == 0 else None)
ax.set(xticks=x, xticklabels=resumen['modelo'], xlabel='Modelo de servicio',
       ylabel='Espera media en cola (minutos)')
ax.legend(fontsize=8)
plt.show()

filas_cdf = []
fig_distribucion, ax = plt.subplots(figsize=(7, 4), layout='constrained')
for e, nombre in ESCENARIOS.items():
    denominador = CONFIG['replicas']*CONFIG['medicion']
    probabilidades = conteos_totales[e]/denominador
    masa_cola = 1-float(probabilidades[-1])
    print(f'{nombre}: masa sobre {UMBRALES[-1]:g} minutos = {masa_cola:.6%}')
    ax.plot(UMBRALES, probabilidades, color=COLORES[e], linestyle=ESTILOS[e],
            marker=MARCAS[e], markevery=40, markersize=4, label=nombre)
    filas_cdf.extend({'escenario_id': e, 'umbral_minutos': float(u), 'conteo': int(v),
                      'denominador': denominador, 'cdf': float(p), 'masa_sobre_ultimo': masa_cola}
                     for u, v, p in zip(UMBRALES, conteos_totales[e], probabilidades))
cdf = pd.DataFrame(filas_cdf)
ax.set(xlabel='Espera en cola (minutos)', ylabel='Fracción con espera ≤ umbral',
       title='Distribución empírica descriptiva: rango finito', ylim=(0, 1.02))
ax.legend(fontsize=8)
plt.show()

fig_delta, ax = plt.subplots(figsize=(7, 4), layout='constrained')
ax.bar(x, 2*resumen['tolerancia_minutos'], bottom=-resumen['tolerancia_minutos'],
       width=0.6, color='0.92', edgecolor='0.6', linewidth=0.5,
       label='Tolerancia práctica: ±2 % de la teoría')
for i, fila in resumen.reset_index(drop=True).iterrows():
    ax.errorbar(i, fila['media_delta'], yerr=fila['h_delta_95'], fmt=MARCAS[i],
                color=COLORES[i], capsize=4, linewidth=0.8,
                label='Diferencia: IC aproximado 95 %' if i == 0 else None)
ax.axhline(0, color='0.5', linewidth=0.6, linestyle='--')
ax.set(xticks=x, xticklabels=resumen['modelo'], xlabel='Modelo de servicio',
       ylabel='Espera alternativa − principal (minutos)')
ax.legend(fontsize=8)
plt.show()

## 9. Lectura crítica pendiente de la ejecución
Antes de redactar resultados, responder con los valores de las tablas: ¿qué precisión se alcanzó en cada escenario?, ¿cuál fue el error con signo frente a la teoría?, ¿qué indica y qué no indica la comparación entre ventanas? Si algún diagnóstico falla, declararlo y conservarlo, sin repetir hasta obtener una salida favorable.

El inicio vacío puede afectar ambas ventanas de modo similar. Los intervalos son aproximaciones entre réplicas, condicionadas a los supuestos y a este presupuesto. La simulación ilustra una comparación analítica; no demuestra una regla general basada solo en varianzas para cualquier disciplina, red o distribución. No extrapolar a servicios reales sin datos. La inspección visual de figuras y la ejecución completa en Colab siguen pendientes hasta realizarlas; este cuaderno no incluye una conclusión experimental preescrita.

## 10. Exportación trazable en la misma carpeta
Se exportan `summary.csv`, `cdf.csv` y cuatro PNG editables desde el código anterior.
`config.csv`, `metadata.json`, `metricas_replicas.csv` y `cdf_replicas.csv` ya existen desde
el cálculo incremental. Los CSV incrementales conservan la etiqueta parcial: solo
`metadata.json` con estado completo certifica que se terminaron las exportaciones.
Si falla una escritura o se interrumpe el cálculo, el estado sigue siendo parcial.
No se genera PDF, no se descarga automáticamente ni se publica. Guardar también una
copia del notebook con sus salidas reales; compartirla es una acción manual del usuario.

In [ ]:
validar_configuracion(CONFIG)
assert len(metricas) == len(ESCENARIOS)*CONFIG['replicas']
assert len(resumen) == len(ESCENARIOS)
assert np.isfinite(metricas[['Y', 'Y_alternativa', 'delta', 'media_servicio', 'varianza_servicio']]).all().all()
for nombre, tabla in [('summary.csv', resumen), ('cdf.csv', cdf)]:
    with ruta_archivo(destino, nombre).open('x', newline='', encoding='utf-8') as archivo:
        tabla.to_csv(archivo, index=False)
for nombre, figura in [('servicios.png', fig_servicios), ('esperas.png', fig_medias),
                       ('distribucion_esperas.png', fig_distribucion), ('inicializacion.png', fig_delta)]:
    with ruta_archivo(destino, nombre).open('xb') as archivo:
        figura.savefig(archivo, format='png', dpi=200, facecolor='white')
guardar_estado(destino, metadatos, 'completo; productos exportados', len(metricas))
print(f'Ubicación absoluta: {destino}')
print('Archivos: config.csv, metadata.json, metricas_replicas.csv, cdf_replicas.csv, summary.csv, cdf.csv, servicios.png, esperas.png, distribucion_esperas.png, inicializacion.png')